# 02 – Macro Data Preparation

Goal: Load, clean, align and validate macroeconomic indicators for use as features in the XGBoost model.
- DE: Destatis 42151-0004 (Orders Received Domestic), 42153-0001 (Production Index)
- US: FRED DGORDER (Durable Goods Orders), FRED INDPRO (Industrial Production Index)

Steps:
1. Load & clean all four macro series (2018+)
2. Merge into a single monthly DataFrame
3. ADF stationarity tests
4. CCF analysis vs. revenue per product (Accessories, Compressors)
5. Save processed output to `data/processed/macro_features.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, ccf

sns.set_theme(style='whitegrid')

DATA_RAW = '../data/raw/'
DATA_PROCESSED = '../data/processed/'

---
## 1. Load & Clean Macro Data

In [ ]:
# DE – Orders Received Index (Destatis 42151-0004)
de_dgorder = pd.read_csv(DATA_RAW + 'DGE_DE_DGORDER.csv', skiprows=1)
de_dgorder = de_dgorder.rename(columns={
    'X13_JDemetra__kalender__und_saisonbereinigt': 'de_orders_index'
})
de_dgorder['date'] = pd.to_datetime(de_dgorder['DateId'], format='%Y-%m')
de_dgorder = de_dgorder[['date', 'de_orders_index']].dropna()
de_dgorder = de_dgorder[de_dgorder['date'] >= '2018-01-01'].reset_index(drop=True)

print('DE DGORDER shape:', de_dgorder.shape)
print('Date range:', de_dgorder['date'].min(), '–', de_dgorder['date'].max())
de_dgorder.head()

In [ ]:
# DE – Industrial Production Index (Destatis 42153-0001)
de_indpro = pd.read_csv(DATA_RAW + 'DGE_DE_INDPRO.csv', skiprows=1)
de_indpro = de_indpro.rename(columns={
    'X13_JDemetra___kalender__und_saisonbereinigt': 'de_production_index'
})
de_indpro['date'] = pd.to_datetime(de_indpro['DateId'], format='%Y-%m')
de_indpro = de_indpro[['date', 'de_production_index']].dropna()
de_indpro = de_indpro[de_indpro['date'] >= '2018-01-01'].reset_index(drop=True)

print('DE INDPRO shape:', de_indpro.shape)
print('Date range:', de_indpro['date'].min(), '–', de_indpro['date'].max())
de_indpro.head()

In [ ]:
# US – Durable Goods Orders (FRED DGORDER)
us_dgorder = pd.read_csv(DATA_RAW + 'DGE_US_DGORDER.csv', skiprows=1)
us_dgorder['date'] = pd.to_datetime(us_dgorder['observation_dateDATE'])
us_dgorder = us_dgorder[['date', 'DGORDER']].rename(columns={'DGORDER': 'us_durable_goods_orders_musd'})
us_dgorder = us_dgorder[us_dgorder['date'] >= '2018-01-01'].reset_index(drop=True)

print('US DGORDER shape:', us_dgorder.shape)
print('Date range:', us_dgorder['date'].min(), '–', us_dgorder['date'].max())
us_dgorder.head()

In [ ]:
# US – Industrial Production Index (FRED INDPRO)
us_indpro = pd.read_csv(DATA_RAW + 'DGE_US_INDPRO.csv', skiprows=1)
us_indpro['date'] = pd.to_datetime(us_indpro['observation_dateDATE'])
us_indpro = us_indpro[['date', 'INDPRO']].rename(columns={'INDPRO': 'us_production_index'})
us_indpro = us_indpro[us_indpro['date'] >= '2018-01-01'].reset_index(drop=True)

print('US INDPRO shape:', us_indpro.shape)
print('Date range:', us_indpro['date'].min(), '–', us_indpro['date'].max())
us_indpro.head()

In [ ]:
# Merge all four macro series on date (inner join – keep only months present in all series)
macro = (
    de_dgorder
    .merge(de_indpro, on='date', how='inner')
    .merge(us_dgorder, on='date', how='inner')
    .merge(us_indpro, on='date', how='inner')
    .sort_values('date')
    .reset_index(drop=True)
)

print('Merged macro shape:', macro.shape)
print('Date range:', macro['date'].min(), '–', macro['date'].max())
print('\nMissing values:')
print(macro.isnull().sum())
macro.head()

In [ ]:
# Plot all four macro series (2018+)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

series_cfg = [
    ('de_orders_index',             'DE – Orders Received Index',          'steelblue',  'Index'),
    ('de_production_index',         'DE – Industrial Production Index',    'darkorange', 'Index'),
    ('us_durable_goods_orders_musd','US – Durable Goods Orders',           'seagreen',   'USD Million'),
    ('us_production_index',         'US – Industrial Production Index',    'crimson',    'Index (2017=100)'),
]

for ax, (col, title, color, ylabel) in zip(axes, series_cfg):
    ax.plot(macro['date'], macro[col], color=color)
    ax.set_title(title)
    ax.set_xlabel('Month')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Macro Indicators (2018+)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 2. ADF Stationarity Tests

The Augmented Dickey-Fuller (ADF) test checks for a unit root (non-stationarity).  
- H₀: Series has a unit root (non-stationary)  
- If p-value < 0.05 → reject H₀ → series is stationary  

Non-stationary series should be differenced before computing lag features.

In [ ]:
def run_adf(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'Series': name,
        'ADF Statistic': round(result[0], 4),
        'p-value': round(result[1], 4),
        'Lags Used': result[2],
        'Stationary (p<0.05)': 'Yes' if result[1] < 0.05 else 'No'
    }

adf_results = pd.DataFrame([
    run_adf(macro['de_orders_index'],              'DE Orders Received Index'),
    run_adf(macro['de_production_index'],          'DE Industrial Production Index'),
    run_adf(macro['us_durable_goods_orders_musd'], 'US Durable Goods Orders'),
    run_adf(macro['us_production_index'],          'US Industrial Production Index'),
])

adf_results

---
## 3. Load Revenue Data

Revenue is loaded per product to enable product-level CCF analysis.  
Only the overlapping period with macro data (2022-10 – 2025-12) will be used for CCF.

In [ ]:
# Load Net Value and parse to monthly per-product series
net_value = pd.read_excel(DATA_RAW + 'DGE_Revenue_Planning_Actuals_Net_Value_US.xlsx', sheet_name=0)
net_value = net_value.rename(columns={'Unnamed: 4': 'net_value_usd'})
net_value['date'] = pd.to_datetime(net_value['Date'].astype(str), format='%Y%m')
net_value = net_value[['date', 'Product', 'net_value_usd']].dropna()

# Pivot to wide format: one column per product
revenue = net_value.pivot_table(index='date', columns='Product', values='net_value_usd').reset_index()
revenue.columns.name = None
revenue = revenue.sort_values('date').reset_index(drop=True)

print('Revenue shape:', revenue.shape)
print('Products:', [c for c in revenue.columns if c != 'date'])
print('Date range:', revenue['date'].min(), '–', revenue['date'].max())
revenue.head()

In [ ]:
# Merge revenue with macro on overlapping dates
ccf_data = macro.merge(revenue, on='date', how='inner').sort_values('date').reset_index(drop=True)

print('CCF dataset shape:', ccf_data.shape)
print('Date range:', ccf_data['date'].min(), '–', ccf_data['date'].max())
ccf_data.head()

---
## 4. CCF Analysis – Macro vs. Revenue per Product

The Cross-Correlation Function (CCF) measures how well each macro indicator  
at lag *k* predicts revenue at time *t* (i.e. macro leads revenue by *k* months).  

- Lag k → macro at t-k correlates with revenue at t  
- The lag with the highest absolute correlation is the optimal feature lag  
- Dashed gray lines mark the 95% confidence interval (±1.96/√n)

**Note on stationarity:** ADF results show that DE Industrial Production Index,  
US Durable Goods Orders, and US Industrial Production Index are non-stationary.  
Their first difference (monthly change) is used for CCF to avoid spurious correlations.  
DE Orders Received Index is already stationary and used as-is.

In [ ]:
# col, label, is_stationary (True = use level, False = use first difference)
MACRO_COLS = [
    ('de_orders_index',              'DE Orders Received Index',         True),
    ('de_production_index',          'DE Industrial Production Index',   False),
    ('us_durable_goods_orders_musd', 'US Durable Goods Orders',         False),
    ('us_production_index',          'US Industrial Production Index',   False),
]

products = [c for c in ccf_data.columns if c not in ['date'] + [m[0] for m in MACRO_COLS]]
N_LAGS = 12

print('Products found:', products)

In [ ]:
# CCF plots: one figure per product, one subplot per macro indicator
optimal_lags = []

for product in products:
    fig, axes = plt.subplots(1, len(MACRO_COLS), figsize=(16, 4))
    fig.suptitle(f'CCF – Macro vs. Revenue: {product}', fontsize=13)

    rev_series = ccf_data[product].values.astype(float)

    for ax, (col, label, is_stationary) in zip(axes, MACRO_COLS):
        macro_series = ccf_data[col].values.astype(float)

        # Use first difference for non-stationary series
        if not is_stationary:
            macro_series = np.diff(macro_series, prepend=np.nan)
            rev_plot     = np.diff(rev_series,   prepend=np.nan)
            # Drop the first NaN row introduced by diff
            valid = ~(np.isnan(macro_series) | np.isnan(rev_plot))
            macro_series = macro_series[valid]
            rev_plot     = rev_plot[valid]
            diff_label   = f'{label} (Δ)'
        else:
            rev_plot   = rev_series.copy()
            diff_label = label

        # Standardize both series before CCF
        rev_std   = (rev_plot     - rev_plot.mean())     / rev_plot.std()
        macro_std = (macro_series - macro_series.mean()) / macro_series.std()

        # ccf(x, y): correlation between x[t] and y[t-k] → macro leads revenue
        ccf_values = ccf(rev_std, macro_std, nlags=N_LAGS, alpha=None)

        # Derive lags from actual output length (statsmodels may return nlags, not nlags+1)
        lags = np.arange(len(ccf_values))
        optimal_lag = int(lags[np.argmax(np.abs(ccf_values))])
        optimal_lags.append({
            'Product': product,
            'Macro': diff_label,
            'Optimal Lag (months)': optimal_lag,
            'CCF at Optimal Lag': round(ccf_values[optimal_lag], 4)
        })

        ax.bar(lags, ccf_values, color='steelblue', alpha=0.7)
        ax.axvline(optimal_lag, color='crimson', linestyle='--', linewidth=1.2,
                   label=f'Optimal lag: {optimal_lag}m')
        ax.axhline(0, color='black', linewidth=0.8)

        # 95% confidence bounds (approx. ±1.96/sqrt(n))
        conf = 1.96 / np.sqrt(len(rev_std))
        ax.axhline( conf, color='gray', linestyle=':', linewidth=0.8)
        ax.axhline(-conf, color='gray', linestyle=':', linewidth=0.8)

        ax.set_title(diff_label, fontsize=9)
        ax.set_xlabel('Lag (months)')
        ax.set_ylabel('Correlation')
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

# Summary table
lag_summary = pd.DataFrame(optimal_lags)
print('\nOptimal Lags Summary:')
lag_summary

---
## 5. Save Processed Output

Save the cleaned and merged macro DataFrame (2018+) to `data/processed/macro_features.csv`.  
This file will be loaded in `03_feature_engineering.ipynb` and merged with revenue data.

In [ ]:
import os
os.makedirs(DATA_PROCESSED, exist_ok=True)

output_path = DATA_PROCESSED + 'macro_features.csv'
macro.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Shape: {macro.shape}')
print(f'Columns: {list(macro.columns)}')
macro.tail()